In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [26]:
# Configure the path of the data to be imported
base_path = Path.cwd()
print("current working directory:",base_path)
data_dir = base_path.parent/"data"/"02_cleaned_data"/"cleaned_data"
print("data direcory:", data_dir)
dest_file = base_path.parent/"data"/"02_cleaned_data"/"batting_stats.csv"

current working directory: /home/mohan/Projects/wpl-2026-analysis/notebooks
data direcory: /home/mohan/Projects/wpl-2026-analysis/data/02_cleaned_data/cleaned_data


In [4]:
source_df = pd.read_csv(data_dir)

In [5]:
source_df

,match_id,date,inning,batting_team,bowling_team,over,ball,batter,bowler,non_striker,runs_bat,runs_extras,runs_total,is_wide,is_noball,is_wicket,player_out,wicket_kind
0,1513682,2026-01-09,1,Mumbai Indians,Royal Challengers Bengaluru,0,1,AC Kerr,LK Bell,G Kamalini,0,0,0,0,0,0,NaN,NaN
1,1513682,2026-01-09,1,Mumbai Indians,Royal Challengers Bengaluru,0,2,AC Kerr,LK Bell,G Kamalini,0,0,0,0,0,0,NaN,NaN
2,1513682,2026-01-09,1,Mumbai Indians,Royal Challengers Bengaluru,0,3,AC Kerr,LK Bell,G Kamalini,0,0,0,0,0,0,NaN,NaN
3,1513682,2026-01-09,1,Mumbai Indians,Royal Challengers Bengaluru,0,4,AC Kerr,LK Bell,G Kamalini,0,0,0,0,0,0,NaN,NaN
4,1513682,2026-01-09,1,Mumbai Indians,Royal Challengers Bengaluru,0,5,AC Kerr,LK Bell,G Kamalini,0,0,0,0,0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5245,1513703,2026-02-05,2,Royal Challengers Bengaluru,Delhi Capitals,18,6,RP Yadav,CA Henry,N de Klerk,1,0,1,0,0,0,NaN,NaN
5246,1513703,2026-02-05,2,Royal Challengers Bengaluru,Delhi Capitals,19,1,RP Yadav,N Shree Charani,N de Klerk,1,0,1,0,0,0,NaN,NaN
5247,1513703,2026-02-05,2,Royal Challengers Bengaluru,Delhi Capitals,19,2,N de Klerk,N Shree Charani,RP Yadav,1,0,1,0,0,0,NaN,NaN
5248,1513703,2026-02-05,2,Royal Challengers Bengaluru,Delhi Capitals,19,3,RP Yadav,N Shree Charani,N de Klerk,4,0,4,0,0,0,NaN,NaN


In [6]:
legal_deleveris = source_df[
    source_df['is_wide'] == 0 
]

In [7]:
batting_stats = legal_deleveris.groupby('batter',as_index=False).agg({
    'runs_bat':'sum',
    'ball':'size'
})

In [8]:
batting_stats

,batter,runs_bat,ball
0,A Gardner,244,172
1,A Reddy,38,50
2,A Soni,11,14
3,AB Kaur,139,104
4,AB Sharma,177,137
...,...,...,...
61,SR Patil,28,17
62,Shafali Verma,259,207
63,Shivani Singh,0,2
64,Simran Shaikh,32,25


In [10]:
batter_outs = source_df.groupby('player_out').size()

In [11]:
batting_stats["outs"] = batting_stats["batter"].map(batter_outs).fillna(0).astype(int)

In [12]:
batting_stats

,batter,runs_bat,ball,outs
0,A Gardner,244,172,9
1,A Reddy,38,50,3
2,A Soni,11,14,1
3,AB Kaur,139,104,6
4,AB Sharma,177,137,7
...,...,...,...,...
61,SR Patil,28,17,2
62,Shafali Verma,259,207,10
63,Shivani Singh,0,2,1
64,Simran Shaikh,32,25,2


In [14]:
# batting average
batting_stats['batting_avg'] = np.where(batting_stats['outs']>0,np.round(batting_stats['runs_bat']/batting_stats['outs'],2),batting_stats['runs_bat'])

In [19]:
batting_stats

,batter,runs_bat,ball,outs,batting_avg,strike_rate
0,A Gardner,244,172,9,27.11,141.9
1,A Reddy,38,50,3,12.67,76.0
2,A Soni,11,14,1,11.00,78.6
3,AB Kaur,139,104,6,23.17,133.7
4,AB Sharma,177,137,7,25.29,129.2
...,...,...,...,...,...,...
61,SR Patil,28,17,2,14.00,164.7
62,Shafali Verma,259,207,10,25.90,125.1
63,Shivani Singh,0,2,1,0.00,0.0
64,Simran Shaikh,32,25,2,16.00,128.0


In [18]:
batting_stats['strike_rate'] = np.round(
    np.divide(
    batting_stats['runs_bat'] * 100,
    batting_stats['ball']
),1
)


In [22]:
batting_stats['batting_impact'] = np.round((batting_stats['batting_avg'] * batting_stats['strike_rate']) / 100 , 2)

In [23]:
batting_stats

,batter,runs_bat,ball,outs,batting_avg,strike_rate,batting_impact
0,A Gardner,244,172,9,27.11,141.9,38.47
1,A Reddy,38,50,3,12.67,76.0,9.63
2,A Soni,11,14,1,11.00,78.6,8.65
3,AB Kaur,139,104,6,23.17,133.7,30.98
4,AB Sharma,177,137,7,25.29,129.2,32.67
...,...,...,...,...,...,...,...
61,SR Patil,28,17,2,14.00,164.7,23.06
62,Shafali Verma,259,207,10,25.90,125.1,32.40
63,Shivani Singh,0,2,1,0.00,0.0,0.00
64,Simran Shaikh,32,25,2,16.00,128.0,20.48


In [30]:
batting_stats[batting_stats['ball'] > 30].sort_values(by='batting_impact',ascending=False)

,batter,runs_bat,ball,outs,batting_avg,strike_rate,batting_impact
22,H Kaur,342,227,5,68.40,150.7,103.08
39,NR Sciver-Brunt,321,212,5,64.20,151.4,97.20
53,S Mandhana,377,246,7,53.86,153.3,82.57
42,P Litchfield,243,157,6,40.50,154.8,62.69
37,N de Klerk,133,96,3,44.33,138.5,61.40
30,L Wolvaardt,317,234,7,45.29,135.5,61.37
38,NJ Carey,149,105,4,37.25,141.9,52.86
18,G Wareham,178,123,5,35.60,144.7,51.51
45,RM Ghosh,189,125,6,31.50,151.2,47.63
20,GM Harris,237,133,9,26.33,178.2,46.92


In [28]:
batting_stats.to_csv(dest_file, index=False)